In [4]:
import os
from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

In [5]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

# HuggingFace model

hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

!pip3 install -U ddgs

### Agent
#### 1. 내장 Tool 사용법

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_core.tools import Tool

from langchain_community.tools import ListDirectoryTool, ReadFileTool, WriteFileTool, CopyFileTool, DeleteFileTool

In [ ]:
# search Tool : DuckDuckGoSearch(무료, api 키 발급 x) 

response = watson_llm.invoke("2026년 대한민국 대통령은 누구인가요?")
response.content

'2026년 대한민국 대통령에 대한 정보는 현재로서는 제공할 수 없습니다. 대통령 선거는 5년마다 열리며, 다음 선거는 2027년에 예정되어 있습니다. 따라서 2026년에 대한민국 대통령이 누구인지는 선거 결과에 따라 결정될 것입니다.'

In [4]:
response = watson_llm.invoke("2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?")
response.content

'죄송하지만, 2024년 노벨 물리학상 수상자의 나이를 알려드릴 수 없습니다. 이는 2024년이 아직 끝나지 않았기 때문입니다. 노벨상 수상자는 매년 10월에 발표되며, 수상자의 나이는 그 시점에서 확인할 수 있습니다.'

In [ ]:
tools = DuckDuckGoSearchRun()

agent = create_agent(
    model=watson_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?"
            }
        ]
    }
)


In [ ]:
tools = DuckDuckGoSearchRun()

agent = create_agent(
    model=watson_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2026년 대한민국 대통령은 누구인가요?"
            }
        ]
    }
)


In [12]:
def print_result(result):
    for i, msg in enumerate(result['messages']):
        print(f"\n===== {i} ====")
        print(type(msg).__name__)

        if hasattr(msg, "content"):
            print(msg.content)

        if hasattr(msg, "tool_calls"):
            print(msg.tool_calls)

In [ ]:
print_result(result)

- 구글 웹서치
    - http://serper.dev/ 회원가입 -> API Key 발급
    

In [ ]:
search = GoogleSerperAPIWrapper()

search_tool = Tool(
    name="google_search",
    func=search.run,
    description="""
    Google  검색으로 최신 정보를 찾습니다.
    뉴스, 트렌드 최신 기술 동향 검색에 사용하세요.
    """
)
agent = create_agent(
    model=hugging_llm,
    tools=[search_tool],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?"
            }
        ]
    }
)

In [38]:
agent = create_agent(
    model=hugging_llm,
    tools=[search_tool],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"2026년 대한민국 대통령은 누구인가요?"
            }
        ]
    }
)

In [41]:
print_result(result)


==== 0 ====
HumanMessage
Content: 2024년 노벨 물리학상 수상자의 나이는 몇 살인가요?

==== 1 ====
AIMessage
Content: 현재 2024년 노벨 물리학상 수상자에 대한 정보는 없습니다. 노벨상은 매년 10월 중순경에 발표되므로, 아직 공식 발표가 이루어지지 않았습니다. 최근의 수상자 정보를 제공해 드리겠습니다.

최근 2023년 노벨 물리학상 수상자의 나이를 확인해 보겠습니다.

🔧 Tool Calls Detected:
  - Tool Name: google_search
    Arguments: {'__arg1': '2023 노벨 물리학상 수상자 나이'}
    Call ID: call_iquhshpqgb6yvpoba4hwf1fe

==== 2 ====
ToolMessage
Content: 스웨덴 왕립과학원 노벨상위원회는 3일 2023년 노벨물리학상 수상자로 피에르 아고스티니 미국 오하이오주립대 교수, 페렌츠 크라우스 독일막스플랑크양자광학연구소·뮌헨대 교수(61), 앤 륄리에 스웨덴 룬드대 교수(65)를 선정했다고 발표했다.

==== 3 ====
AIMessage
Content: 2023년 노벨 물리학상 수상자들의 나이는 다음과 같습니다:

- 피에르 아고스티니: 61세
- 페렌츠 크라우스: 61세
- 앤 륄리에: 65세


- Wikipedia

In [42]:
!pip3 install wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=47bad6d8ee7937c798b477418fc8242d34b704f13a3331581d61e502ae3171fe
  Stored in directory: c:\users\soldesk\appdata\local\pip\cache\wheels\63\47\7c\a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
tools = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

agent = create_agent(
    model=qwen_llm,
    tools=[tools],
    system_prompt="You are a helpful assistant"
)

result =  agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"앨런 튜링은 누구인가요? 한국어로 대답해주세요"
            }
        ]
    }
)
print_result(result)


==== 0 ====
HumanMessage
Content: 앨런 튜링은 누구인가요? 한국어로 대답해주세요

==== 1 ====
AIMessage
Content: 앨런 튜링 (Alan Turing) 은 1912 년 영국에서 태어난 수학자이자 컴퓨터 과학자로, 현대 컴퓨팅 이론의 아버지로 평가받는 인물입니다. 주요 업적은 다음과 같습니다:

**생애와 배경:**
- 영국의 브리스톨 출신으로, 케임브리지 대학교를 졸업했습니다.
- 2 차 세계대전 당시 영국 해군에서 암호학 연구에 참여했으며, 독일의 Enigma 기계를 해독하는 데 결정적인 역할을 했습니다.

**주요 업적:**
1. **컴퓨터 과학 이론**: 튜링 기계 (Turing Machine) 를 제안하여 현대 컴퓨터의 기본 원리를 정립했습니다.
2. **인공지능 연구**: 1950 년에 "Computing Machinery and Intelligence"라는 논문에서 인공지능이 존재할 수 있다는 논의를 시작했으며, 이후 '튜링 테스트'를 제시했습니다.
3. **암호학**: Bletchley Park 에서 독일의 Enigma 암호 해독을 주도하여 연합군의 승리에 크게 기여했습니다.

**사망:**
- 1952 년에 동성애 혐의로 기소되어 유죄 판결을 받고, 화학 약품으로 자살한 것으로 알려져 있습니다 (실제로는 스스로를 죽였다는 설이 지배적입니다).

튜링은 과학과 기술의 발전에 지대한 공헌을 한 인물로 평가받으며, 1954 년 영국에서 가장 위대한 수학자로 선정되었습니다.


- 파일관련 tool

In [ ]:
tools = [ReadFileTool(),WriteFileTool(),CopyFileTool(),ListDirectoryTool(), DeleteFileTool()]
agent = create_agent(
    model=watson_llm,
    tools=tools
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                # "content":"현재 파일 목록 보여줘"
                # "content":"food3.py 파일의 내용 보여줘"
                # "content":"현재 파일목록을 summary.text로 생성해줘"
                "content":"현재 파일목록을 summary2.text로 생성하고 복사해서 summary.csv파일로 저장후 summary.text 삭제해줘"
            }
        ]
    }
)

print_result(result)


==== 0 ====
HumanMessage
Content: 현재 파일목록을 summary2.text로 생성하고 복사해서 summary.csv파일로 저장후 summary.text 삭제해줘

==== 1 ====
AIMessage
🔧 Tool Calls Detected:
  - Tool Name: list_directory
    Arguments: {'dir_path': '.'}
    Call ID: chatcmpl-tool-9258cb278f8c51b5

==== 2 ====
ToolMessage
Content: .env
.venv
data
db
finder.py
food.py
food2.py
food3.py
langachain3.ipynb
langchain-agent.ipynb
langchain.ipynb
langchain2.ipynb
news.py
news2.py
ollama.ipynb
ollama.js
pdfrag.py
pdf_분쟁조정.ipynb
rag_company.py
rag_naver.ipynb
review.py
summary.txt

==== 3 ====
AIMessage
🔧 Tool Calls Detected:
  - Tool Name: write_file
    Arguments: {'file_path': 'summary2.txt', 'text': '.env\n.venv\ndata\ndb\nfinder.py\nfood.py\nfood2.py\nfood3.py\nlangachain3.ipynb\nlangchain-agent.ipynb\nlangchain.ipynb\nlangchain2.ipynb\nnews.py\nnews2.py\nollama.ipynb\nollama.js\npdfrag.py\npdf_부잡조정.ipynb\nrag_company.py\nrag_naver.ipynb\nreview.py\nsummary.txt'}
    Call ID: chatcmpl-tool-854b49b930d6aaf5

==== 4 ====
ToolMessa

In [4]:
import sqlite3

In [10]:
conn = sqlite3.connect("./db/company.db")

cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS employee")
cursor.execute("""
CREATE TABLE employee(id integer primary key autoincrement, name text, salary integer)
""")
cursor.execute("""
INSERT INTO employee(name, salary)
VALUES
('Kim', 4000),               
('Lee', 5000),              
('Park', 6000)               
""")

conn.commit()
conn.close()

In [8]:
!pip3 install sqlalchemy

In [2]:
# sqlalchemy:Python 에서 데이터베이스를 편하게 다루기위한 라이브러리

from langchain_community.utilities import SQLDatabase

from langchain_community.agent_toolkits import SQLDatabaseToolkit

C:\Users\soldesk\AppData\Local\Temp\ipykernel_14032\386282781.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase
c:\souce\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:




db = SQLDatabase.from_uri("sqlite:///db/company.db")

print(db.run("select * from employee"))

[(1, 'Kim', 4000), (2, 'Lee', 5000), (3, 'Park', 6000)]


In [ ]:


toolkit = SQLDatabaseToolkit(db=db, llm=watson_llm)
tools = toolkit.get_tools()

agent = create_agent(
    model=watson_llm,
    tools=tools,
    system_prompt="You are a database assistant."
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"직원 평균 급여는 얼마인가요?"
            }
        ]
    }
)

print_result(result)


===== 0 ====
HumanMessage
직원 평균 급여는 얼마인가요?

===== 1 ====
AIMessage

[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'chatcmpl-tool-9cf2d081fdcef0f5', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
employee

===== 3 ====
AIMessage

[{'name': 'sql_db_schema', 'args': {'table_names': 'employee'}, 'id': 'chatcmpl-tool-963aeb6355632743', 'type': 'tool_call'}]

===== 4 ====
ToolMessage

CREATE TABLE employee (
	id INTEGER, 
	name TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employee table:
id	name	salary
1	Kim	4000
2	Lee	5000
3	Park	6000
*/

===== 5 ====
AIMessage

[{'name': 'sql_db_query', 'args': {'query': 'SELECT AVG(salary) AS avg_salary FROM employee;'}, 'id': 'chatcmpl-tool-b79acc56ac177750', 'type': 'tool_call'}]

===== 6 ====
ToolMessage
[(5000.0,)]

===== 7 ====
AIMessage
직원들의 평균 급여는 **5,000**입니다.
[]


- calculator
    - 계산

In [14]:
!pip3 install numexpr

In [6]:
from langchain_classic.chains import LLMMathChain

In [ ]:


llm_math = LLMMathChain.from_llm(llm=watson_llm, verbose=True)

calc_tool = Tool(name="calculator", func=llm_math.run, description="수학 계산이 필요할때 사용합니다. 시장 규모 계산, 성장률 계산, 단위 변환 등에 사용하세요.")

calc_tool.invoke("1200억 달러를 원화로 환산(1달러=1,543원)")



> Entering new LLMMathChain chain...
1200억 달러를 원화로 환산(1달러=1,543원)Question: 1200억 달러를 원화로 환산(1달러=1,543원)
```text
120000000000 * 1543
```
...numexpr.evaluate("120000000000 * 1543")...
```output
185160000000000
```
Answer: 185160000000000 원
> Finished chain.


'Answer:  185160000000000 원'

#### 커스텀 Tool
- description 작성
    - 기능 설명, 사용 조건, 입력값 형식, 출력값 형식, 필수 or 선택 명시

- 작성 방법
    - @tool 데코레이터 (주로 사용됨)
    - StructedTool
    - BaseTool

In [1]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

In [ ]:


# 함수 구현

@tool
def calculator(expression):
    """
    수식을 계산 합니다.
    사칙 연산에 사용하세요.
    """
    return str(eval(expression))

calculator.invoke("123 * 567")


'69741'

In [21]:
agent = create_agent(
    model=watson_llm,
    tools=[calculator],
    system_prompt="You are a calculator assistant."
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"123 곱하기 789는??"
            }
        ]
    }
)

print_result(result)


===== 0 ====
HumanMessage
123 곱하기 789는??

===== 1 ====
AIMessage

[{'name': 'calculator', 'args': {'expression': '123 * 789'}, 'id': 'chatcmpl-tool-812694eb46f4f380', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
97047

===== 3 ====
AIMessage
123 곱하기 789는 97047입니다.
[]


In [ ]:


class ExchangeRateInput(BaseModel):
    from_currency:str=Field(description='변환할 통화 코드 (예: USD, EUR, JPY,KRW)')
    to_currency:str=Field(description='변환될 통화 코드 (예: USD, KRW)')
    amount:float = Field(default=1.0, description="변환할 금액(기본값:1)")

@tool(args_schema=ExchangeRateInput)
def get_exchange_rate(from_currency, to_currency,amount):
    """
    실시간 환율을 조회하고 금액을 변환합니다.
    USD, EUR, JYP, KRW 통화 간 환율 계산에 사용하세요.
    해외 결제, 환전, 가격 비교 질문에 사용합니다.
    """
    mock_rates={
        ('USD','KRW'):1540.0,
        ('EUR','KRW'):1440.0,
        ('JPY','KRW'):9.2,
        ('KRW','USD'):0.00064,
    }
    rate = mock_rates.get((from_currency,to_currency))
    if not rate:
        return f"{from_currency}->{to_currency}"
    converted = amount * rate

    return f"{amount}{from_currency}= {converted:.2f}{to_currency} (환율:{rate})"

In [23]:
agent = create_agent(
    model=watson_llm,
    tools=[get_exchange_rate]
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"1달러는 원화로 얼마인가요??"
            }
        ]
    }
)

print_result(result)


===== 0 ====
HumanMessage
1달러는 원화로 얼마인가요??

===== 1 ====
AIMessage

[{'name': 'get_exchange_rate', 'args': {'amount': 1, 'from_currency': 'USD', 'to_currency': 'KRW'}, 'id': 'chatcmpl-tool-b6d27d1006ae295a', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
1.0USD= 1540.00KRW (환율:1540.0)

===== 3 ====
AIMessage
1달러는 현재 **1,540원** 정도에 해당합니다. (환율: 1 USD = 1,540 KRW)
[]


In [ ]:


def multiply(a:int, b:int)->int:
    "두 숫자를 곱합니다."
    return a * b

# 이미 존재하는 python 함수를 Tool로 변환
multi_tool = StructuredTool.from_function(func=multiply)

agent = create_agent(
    model=watson_llm,
    tools=[multiply]
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"12 곱하기 34는??"
            }
        ]
    }
)

print_result(result)



===== 0 ====
HumanMessage
12 곱하기 34는??

===== 1 ====
AIMessage

[{'name': 'multiply', 'args': {'a': 12, 'b': 34}, 'id': 'chatcmpl-tool-996da207c627faac', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
408

===== 3 ====
AIMessage
12를 34로 곱하면 **408**입니다.
[]


In [25]:
def calculate_bmi(height:float,weight:float)->str:
    """
    heught(cm)과 weight(kg)을 입력 받아 BMI 계산
    """
    bmi = weight/((height/100)**2)
    return f"BMI={bmi:.2f}"



In [27]:
bmi_tool = StructuredTool.from_function(calculate_bmi)

# agent 등록
agent = create_agent(
    model=watson_llm,
    tools=[bmi_tool]
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"180cm 80kg인 사람의 bmi는??"
            }
        ]
    }
)

print_result(result)


===== 0 ====
HumanMessage
180cm 80kg인 사람의 bmi는??

===== 1 ====
AIMessage

[{'name': 'calculate_bmi', 'args': {'height': 180, 'weight': 80}, 'id': 'chatcmpl-tool-8ec58fa3c69ba5ff', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
BMI=24.69

===== 3 ====
AIMessage
180cm, 80kg인 사람의 BMI는 **24.69**입니다. 이는 정상 체중 범위에 해당합니다.
[]


In [10]:
from langchain_core.tools import BaseTool

class WeatherInput(BaseModel):
    city:str=Field(description="날씨를 조회할 도시명")

class WeatherTool(BaseTool):
    name:str = "weather"
    description:str = "특정 도시의 날씨를 조회합니다."

    args_schema:type[BaseModel]=WeatherInput
    def _run(self, city:str):

        Weather_data = {
            "서울":"맑음, 30도",
            "부산":"맑음, 32도",
            "제주":"비, 20도"
        }

        return Weather_data.get(city, f"{city} 날씨 정보 없음")

In [11]:
weather_tool = WeatherTool()

agent = create_agent(
    model=watson_llm,
    tools=[weather_tool]
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"서울 날씨는??"
            }
        ]
    }
)

print_result(result)

NameError: name 'print_result' is not defined

### Retriever Tool

In [14]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_classic.tools.retriever import create_retriever_tool

docs = [
    Document(page_content="연차는 입사 1년 미만은 월 1개 발생합니다."),
    Document(page_content="병가는 연 30일 까지 사용 가능합니다."),
    Document(page_content="퇴직금은 1년 이상 근무 시 지급됩니다."),
]

vectorstore = Chroma.from_documents(docs, embedding=watsonx_enbedding)

retriever = vectorstore.as_retriever()

# retriever 를 tool로 등록
retriever_tool = create_retriever_tool(retriever, name="company_policy", description="회사 인사 규정을 검색합니다.")

In [ ]:
agent = create_agent(
    model=watson_llm,
    tools=[retriever_tool]
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"퇴직금은 언제부터 받을 수 있는가??"
            }
        ]
    }
)

# print_result(result)

https://smith.langchain.com/

### 리서치 자동화 Agent 구현
- 입력 :  리서치 주제
- tool
    - 1. 리서치 주제에 대해 웹검색으로 최신 동향 수집 tool /google
    - 2. 최신 동향 요약 tool
    - 3. 수치 데이터가 있다면 계산기로 검증/변환 tool
    - 4. Markdown 형식의 문서로 저장 tool 

In [9]:
# 웹검색 tool(google)
search = GoogleSerperAPIWrapper()

search_tool = Tool(
    name="google_search",
    func=search.run,
    description="""
    Google  검색으로 최신 정보를 찾습니다.
    뉴스, 트렌드 최신 기술 동향 검색에 사용하세요.
    """
)
result = search_tool.run("AI Semicinductor Market 2025")

In [10]:
@tool
def trend_analysis_tool(text):
    """
    검색 결과에서 최신 트렌드를 분석합니다.
    """

    prompt = f""""
    아래 검색 결과를 분석하여

    1. 핵심 트랜드
    2. 주요 기업
    3. 성장 요인
    4. 위험 요소

    를 정리하세요.

    검색결과:
    {text}
    """
    return watson_llm.invoke(prompt).content

trend_analysis_tool.invoke(result)

'Based on the provided search results, here is the analysis:\n\n### 1. 핵심 트랜드 (Key Trends)\n- **AI의 성장**: 인공지능(AI) 기술의 발전이 반도체 시장의 성장을 주도하고 있습니다. Nvidia의 수익이 65% 증가한 것은 AI의 중요성을 강조합니다.\n- **시장 성장**: 2025년 반도체 시장은 2024년 대비 25.6% 증가하여, 2021년 COVID-19 회복 기간 이후 가장 강력한 성장세를 보였습니다.\n\n### 2. 주요 기업 (Major Companies)\n- **Nvidia**: AI 기술의 발전에 크게 기여하고 있으며, 수익이 65% 증가하여 주목받고 있습니다.\n\n### 3. 성장 요인 (Growth Factors)\n- **AI 기술의 발전**: AI 기술의 발전이 반도체 시장의 성장을 주도하고 있습니다. 특히, Nvidia의 성장이 이를 잘 보여줍니다.\n- **시장 수요 증가**: 반도체 시장의 전반적인 수요 증가가 시장 성장에 기여하고 있습니다.\n\n### 4. 위험 요소 (Risk Factors)\n- **시장 변동성**: 반도체 시장은 기술 발전과 수요 변화에 민감하게 반응할 수 있어, 시장 변동성이 위험 요소로 작용할 수 있습니다.\n- **기술 의존성**: AI 기술에 대한 과도한 의존은 기술 변화에 따른 위험을 초래할 수 있습니다. 기술 변화에 빠르게 적응하지 못하면 경쟁력이 떨어질 수 있습니다.\n\n이 분석은 제공된 검색 결과를 기반으로 하며, 추가적인 데이터와 상황에 따라 변동될 수 있습니다.'

In [11]:
import numexpr
# 계산기
@tool
def calculator_tool(expression):
    """
    수학 계산용 Tool

    사용 예:
    999*1530
    (250/120)**(1/6)-1
    10000000/1350

    CAGR 계산
    환율 계산
    시장 규모 계산
    """
    result = numexpr.evaluate(expression)

    return str(result.item())

calculator_tool.invoke("2+3*(4-1)")

'11'

In [12]:
from datetime import datetime

# @tool
# def save_report_tool(content):
def save_report(content):
    """
    리서치 결과를 Markdown 파일로 저장 합니다.
    최종 리포트 작성이 완료 되었을 때 사용하세요.
    입력 : Markdown 형식의 리포트 전체 내용
    """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename= f"research_report{timestamp}.md"
    with open(filename, 'w',encoding='utf-8') as f:
        f.write(content)
    
    return f"리포트 작성완료 : {filename}"

content = """
리서치 결과를 Markdown 파일로 저장 합니다.
최종 리포트 작성이 완료 되었을 때 사용하세요.
입력 : Markdown 형식의 리포트 전체 내용
"""
# save_report_tool.invoke(content)

In [13]:
tools =[search_tool, trend_analysis_tool,calculator_tool]

system_prompt = """
당신은 전문 리서치 어시스던트입니다.

작업순서:
1. search_tool 사용
2. trend_analysis_tool 사용
3. 수치 데이터는 calculator_tool 사용
4. Markdown 리포트 작성
"""

agent = create_agent(
    model=qwen_llm,
    tools=tools,
    system_prompt=system_prompt
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"""
AI 반도체 시장 2026년 전망에 대한 리서치 리포트를 작성해주세요.

주제 :
AI 반도체 시장 2026년 전망

포함 내용:
- 주요기업
- 시장 규모
- 성장률
- 핵심 트랜드
"""
            }
        ]
    }
)

# print(result['messages'])
report = result['messages'][-1].content
filename = save_report(report)
print(f"저장 완료{filename}")

저장 완료리포트 작성완료 : research_report20260605_145631.md
